# organic_ratio — entry notebook


## 1. Git sync

Подтянуть актуальный `main` из `Anton-Filimoncev-azur/organic_ratio` поверх локальной копии. Требует `GIT_PAT` в `.env`.

In [1]:
import os
import subprocess
import base64
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = "/home/jovyan/KEDRO/organic_ratio"
BRANCH = "main"
ORG = "Anton-Filimoncev-azur"
REPO = "organic_ratio"
USER = "Anton-Filimoncev-azur"

os.chdir(PROJECT_ROOT)
print("PWD:", Path().resolve())

load_dotenv()
TOKEN = os.getenv("GIT_PAT")
if not TOKEN:
    raise ValueError("GIT_PAT not found in environment")

auth = base64.b64encode(f"{USER}:{TOKEN}".encode()).decode()

subprocess.run(["git", "rebase", "--abort"], stderr=subprocess.DEVNULL)
subprocess.run(["git", "merge", "--abort"], stderr=subprocess.DEVNULL)
subprocess.run(["git", "reset", "--hard"], check=True)
subprocess.run(["git", "clean", "-fd"], check=True)

subprocess.run(
    [
        "git",
        "-c",
        f"http.extraheader=Authorization: Basic {auth}",
        "fetch",
        f"https://github.com/{ORG}/{REPO}.git",
        BRANCH,
    ],
    check=True,
)
subprocess.run(["git", "reset", "--hard", "FETCH_HEAD"], check=True)

print("Repository synced successfully.")

PWD: /home/jovyan/KEDRO/organic_ratio
HEAD is now at ec1bbe2  work
HEAD is now at 771a167  work
Repository synced successfully.


From https://github.com/Anton-Filimoncev-azur/organic_ratio
 * branch            main       -> FETCH_HEAD


## 2. Env

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## 3. Ingestion (S3 → data/raw/partition/)

In [3]:
# %run run.py

## 4. Per-source preprocessing (raw → data/features/partition/)

In [4]:
# import gc
# %run run_preprocessing.py
# gc.collect()

## 5. Cohort aggregation

`run_cohort_aggregation.py` — мерджит per-source user-grain фичи и роллапит до cohort-grain по ключам из `cohort.keys`. Добавляет `cohort_size` и `n_calendar_days`.

In [5]:
# %run run_cohort_aggregation.py

## 6. Target + features build (model-ready train / test)

`run_target_build.py` за один проход:
1. Считает target (`organic_share`, `total_installs`, `organic_installs`) на грануляции `cohort.keys − media_source`.
2. Реагрегирует все user-grain фичи на ту же грануляцию (политика SUM/MEAN из `aggregate_to_cohort`).
3. Джойнит target + features, пишет:
   - `data/features/targets/targets.parquet` (полный + колонка `split`)
   - `data/train/targets_train.parquet` (split == train, без `split`)
   - `data/test/targets_test.parquet` (split == test, без `split`)

После этого `targets_train.parquet` / `targets_test.parquet` — это уже готовые для модели таблицы.

In [6]:
# import gc
# %run run_target_build.py
# gc.collect()

## 7. Clean (filter small cohorts)

`run_clean.py` — отрезает когорты, где `total_installs < cleaning.min_total_installs` (значение в `parameters.yml`). Пишет `targets_train_clean.parquet` и `targets_test_clean.parquet`.

In [7]:
# import gc
# %run run_clean.py
# gc.collect()

## 8. Baseline (weighted Ridge on logit)

`run_baseline.py` — линейный baseline:
- target: `logit(organic_share)`
- `sklearn.Ridge(alpha=1.0)` с `sample_weight = total_installs`
- preprocessing: numeric → median impute + StandardScaler; `platform` → one-hot; `country_code` → weighted target-encoding (mean от train); `install_date` → drop
- метрики (weighted/unweighted RMSE, MAE, R²) train + test
- top-20 коэффициентов
- сохраняет `data/predictions/baseline_{train,test}.parquet` и `data/plots/baseline_{calibration,coefficients}.png`

In [8]:
%run run_baseline.py

Adding SRC_PATH: /home/jovyan/KEDRO/organic_ratio/src
Loading train: data/train/targets_train_clean.parquet
Loading test:  data/test/targets_test_clean.parquet
train: (14869, 46), test: (2069, 46)

Fitting Ridge on 41 numeric features + country_te + platform OHE

--- Metrics ---
 train: RMSE_w=0.1303  MAE_w=0.0954  R²_w=0.5237  WMAPE=28.32%  | RMSE_u=0.1691  MAE_u=0.1218
  test: RMSE_w=0.1209  MAE_w=0.0911  R²_w=0.5227  WMAPE=28.41%  | RMSE_u=0.1448  MAE_u=0.1090

--- PE distribution (train, n=14,860 non-zero targets) ---
  mean    : +10.7%
  median  : -2.7%
  Q05     : -58.0%
  Q25     : -29.1%
  Q75     : +31.9%
  Q95     : +122.3%
  within ±10%:  18.9%
  within ±20%:  34.9%
  within ±50%:  73.2%

--- PE distribution (test, n=2,069 non-zero targets) ---
  mean    : +13.9%
  median  : -0.9%
  Q05     : -48.7%
  Q25     : -23.1%
  Q75     : +36.5%
  Q95     : +119.3%
  within ±10%:  20.8%
  within ±20%:  37.8%
  within ±50%:  76.9%

--- PE buckets (train, n=14,860) ---
              bu

## 9. PyMC hierarchical model

`run_train.py` — иерархическая байесовская модель:

```
organic_installs[i] ~ Binomial(total_installs[i], p[i])
logit(p[i]) = α + X[i]·β + u_country[c] + u_platform[plat]
u_country  ~ Normal(0, σ_country),  σ_country ~ HalfNormal(1)
u_platform ~ Normal(0, σ_platform), σ_platform ~ HalfNormal(1)
```

Binomial-правдоподобие учитывает размер когорты — `total_installs` есть в самой функции потерь, отдельных весов не нужно.

Параметры сэмплирования в `modeling.pymc` (`parameters.yml`):
- `draws: 500`, `tune: 500`, `chains: 2`, `target_accept: 0.9`
- `nuts_sampler: pymc` (можно `numpyro` для скорости, нужен `pip install numpyro jax`)

Артефакты: `data/models/pymc/{trace.nc, prep.pkl}`, `data/predictions/pymc_{train,test}.parquet`, `data/plots/pymc_{calibration,beta}.png`.

In [ ]:
%run run_train.py

## (опц.) Пакетный запуск экспериментов через CONFIG_OVERRIDE_PATH

In [10]:
# import os
# from pathlib import Path
# from omegaconf import OmegaConf
#
# SWEEP_PATH = Path("conf/batch_training/sweep.yml")
# TMP_DIR = Path("conf/batch_training/.tmp")
#
# sweep_cfg = OmegaConf.load(SWEEP_PATH)
# TMP_DIR.mkdir(parents=True, exist_ok=True)
#
# for exp in sweep_cfg.experiments:
#     run_name = exp.name
#     override_path = TMP_DIR / f"{run_name}.yml"
#     OmegaConf.save(config=exp.overrides, f=override_path)
#     os.environ["CONFIG_OVERRIDE_PATH"] = str(override_path)
#     try:
#         %run run_train.py
#         %run run_eval.py
#     finally:
#         os.environ.pop("CONFIG_OVERRIDE_PATH", None)